In [3]:
# Cell 1: Install and Import Libraries
!pip install folium
!pip install geopy

import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster, MousePosition
from geopy.distance import geodesic
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

# Cell 2: Load Clean Data
df = pd.read_csv('data/spacex_cleaned.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
display(df.head())

# Cell 3: Define Launch Site Coordinates
launch_sites = {
    'CCAFS SLC 40': {'lat': 28.5621, 'lon': -80.577, 'name': 'Cape Canaveral Air Force Station'},
    'CCAFS LC 40': {'lat': 28.5621, 'lon': -80.577, 'name': 'Cape Canaveral Air Force Station'},
    'KSC LC 39A': {'lat': 28.6080, 'lon': -80.6040, 'name': 'Kennedy Space Center'},
    'VAFB SLC 4E': {'lat': 34.6321, 'lon': -120.6100, 'name': 'Vandenberg Air Force Base'},
    'VAFB SLC 4W': {'lat': 34.6321, 'lon': -120.6100, 'name': 'Vandenberg Air Force Base'}
}

print("\nLaunch site coordinates:")
for site, coords in launch_sites.items():
    print(f"  {site}: ({coords['lat']}, {coords['lon']})")

# Cell 4: Add Latitude and Longitude to DataFrame
def get_coordinates(launch_site):
    """Get coordinates for launch site"""
    for site_key, coords in launch_sites.items():
        if site_key in str(launch_site):
            return coords['lat'], coords['lon']
    return 28.5621, -80.577

df[['Latitude', 'Longitude']] = df['LaunchSite'].apply(
    lambda x: pd.Series(get_coordinates(x))
)

print("\nDataFrame with coordinates:")
display(df[['LaunchSite', 'Latitude', 'Longitude']].head())

# Cell 5: MAP 1 - All Launch Sites on Global Map (SLIDE 35)
print("=" * 70)
print("MAP 1: All Launch Sites - Global View")
print("=" * 70)

map1 = folium.Map(
    location=[37.0, -95.7],
    zoom_start=4,
    tiles='OpenStreetMap'
)

unique_sites = df[['LaunchSite', 'Latitude', 'Longitude']].drop_duplicates()

for idx, row in unique_sites.iterrows():
    site_launches = len(df[df['LaunchSite'] == row['LaunchSite']])
    site_successes = df[df['LaunchSite'] == row['LaunchSite']]['Class'].sum()
    success_rate = (site_successes / site_launches * 100) if site_launches > 0 else 0
    
    popup_html = f"""
    <div style="font-family: Arial; font-size: 12px;">
        <b>{row['LaunchSite']}</b><br>
        Total Launches: {site_launches}<br>
        Successful: {site_successes}<br>
        Success Rate: {success_rate:.1f}%<br>
        Coordinates: ({row['Latitude']:.4f}, {row['Longitude']:.4f})
    </div>
    """
    
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=row['LaunchSite'],
        icon=folium.Icon(color='blue', icon='rocket', prefix='fa')
    ).add_to(map1)
    
    folium.Circle(
        location=[row['Latitude'], row['Longitude']],
        radius=site_launches * 1000,
        color='blue',
        fill=True,
        fillColor='lightblue',
        fillOpacity=0.3,
        popup=f'{site_launches} launches'
    ).add_to(map1)

title_html = '''
<div style="position: fixed; 
            top: 10px; 
            left: 50px; 
            width: 400px; 
            height: 50px; 
            background-color: white; 
            border:2px solid grey; 
            z-index:9999; 
            font-size:16px;
            font-weight: bold;
            padding: 10px;
            ">
SpaceX Launch Sites - Global View
</div>
'''
map1.get_root().html.add_child(folium.Element(title_html))

map1.save('folium_map1_global_sites.html')
print("\n[DONE] Map 1 saved: folium_map1_global_sites.html")
print("Screenshot this map for SLIDE 35")
map1

# Cell 6: MAP 2 - Launch Outcomes with Color-Coded Markers (SLIDE 36)
print("\n" + "=" * 70)
print("MAP 2: Launch Outcomes - Color-Coded by Success/Failure")
print("=" * 70)

map2 = folium.Map(
    location=[28.5, -80.6],
    zoom_start=6,
    tiles='OpenStreetMap'
)

for idx, row in df.iterrows():
    if row['Class'] == 1:
        color = 'green'
        icon = 'check'
        outcome_text = 'Success'
    else:
        color = 'red'
        icon = 'times'
        outcome_text = 'Failure'
    
    popup_html = f"""
    <div style="font-family: Arial; font-size: 12px;">
        <b>Flight #{row['FlightNumber']}</b><br>
        Date: {row['Date'].strftime('%Y-%m-%d')}<br>
        Site: {row['LaunchSite']}<br>
        Payload: {row['PayloadMass']:.0f} kg<br>
        Orbit: {row['Orbit_clean']}<br>
        <b style="color: {color};">Outcome: {outcome_text}</b>
    </div>
    """
    
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=6,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"Flight {row['FlightNumber']} - {outcome_text}",
        color=color,
        fillColor=color,
        fillOpacity=0.7,
        weight=2
    ).add_to(map2)

legend_html = '''
<div style="position: fixed; 
            bottom: 50px; 
            right: 50px; 
            width: 180px; 
            height: 120px; 
            background-color: white; 
            border:2px solid grey; 
            z-index:9999; 
            font-size:14px;
            padding: 10px;
            ">
<b>Landing Outcome</b><br>
<i class="fa fa-circle" style="color:green"></i> Successful Landing<br>
<i class="fa fa-circle" style="color:red"></i> Failed Landing<br>
<br>
<small>Click markers for details</small>
</div>
'''
map2.get_root().html.add_child(folium.Element(legend_html))

title_html = '''
<div style="position: fixed; 
            top: 10px; 
            left: 50px; 
            width: 400px; 
            height: 50px; 
            background-color: white; 
            border:2px solid grey; 
            z-index:9999; 
            font-size:16px;
            font-weight: bold;
            padding: 10px;
            ">
SpaceX Launch Outcomes (Success: Green | Failure: Red)
</div>
'''
map2.get_root().html.add_child(folium.Element(title_html))

map2.save('folium_map2_launch_outcomes.html')
print("\n[DONE] Map 2 saved: folium_map2_launch_outcomes.html")
print("Screenshot this map for SLIDE 36")
map2

# Cell 7: MAP 3 - Launch Site with Proximities (SLIDE 37)
print("\n" + "=" * 70)
print("MAP 3: Launch Site Proximities and Distances")
print("=" * 70)

ksc_location = [28.6080, -80.6040]

map3 = folium.Map(
    location=ksc_location,
    zoom_start=12,
    tiles='OpenStreetMap'
)

folium.Marker(
    location=ksc_location,
    popup='<b>Kennedy Space Center LC-39A</b><br>Primary SpaceX Launch Site',
    tooltip='KSC LC-39A',
    icon=folium.Icon(color='red', icon='rocket', prefix='fa')
).add_to(map3)

proximities = {
    'Coastline': [28.6080, -80.5800],
    'Highway (SR 405)': [28.6000, -80.6200],
    'Railway': [28.5900, -80.6300],
    'Visitor Complex': [28.5240, -80.6810],
    'Cape Canaveral SFS': [28.5621, -80.5770]
}

for name, coords in proximities.items():
    distance_km = geodesic(ksc_location, coords).kilometers
    distance_mi = distance_km * 0.621371
    
    popup_html = f"""
    <div style="font-family: Arial; font-size: 12px;">
        <b>{name}</b><br>
        Distance from KSC LC-39A:<br>
        {distance_km:.2f} km ({distance_mi:.2f} miles)
    </div>
    """
    
    folium.Marker(
        location=coords,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f'{name} ({distance_km:.2f} km)',
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(map3)
    
    folium.PolyLine(
        locations=[ksc_location, coords],
        color='red',
        weight=2,
        opacity=0.7,
        popup=f'Distance: {distance_km:.2f} km'
    ).add_to(map3)
    
    print(f"Distance to {name}: {distance_km:.2f} km ({distance_mi:.2f} miles)")

distance_ranges = [5, 10, 20]
colors = ['green', 'yellow', 'orange']

for distance, color in zip(distance_ranges, colors):
    folium.Circle(
        location=ksc_location,
        radius=distance * 1000,
        color=color,
        fill=True,
        fillOpacity=0.1,
        popup=f'{distance} km radius'
    ).add_to(map3)

legend_html = '''
<div style="position: fixed; 
            bottom: 50px; 
            right: 50px; 
            width: 200px; 
            height: 180px; 
            background-color: white; 
            border:2px solid grey; 
            z-index:9999; 
            font-size:12px;
            padding: 10px;
            ">
<b>Distance Ranges from KSC</b><br>
<i class="fa fa-circle" style="color:green"></i> 0-5 km<br>
<i class="fa fa-circle" style="color:yellow"></i> 5-10 km<br>
<i class="fa fa-circle" style="color:orange"></i> 10-20 km<br>
<br>
<i class="fa fa-rocket" style="color:red"></i> Launch Site<br>
<i class="fa fa-info-circle" style="color:blue"></i> Point of Interest<br>
</div>
'''
map3.get_root().html.add_child(folium.Element(legend_html))

title_html = '''
<div style="position: fixed; 
            top: 10px; 
            left: 50px; 
            width: 450px; 
            height: 50px; 
            background-color: white; 
            border:2px solid grey; 
            z-index:9999; 
            font-size:16px;
            font-weight: bold;
            padding: 10px;
            ">
KSC LC-39A Proximities and Distance Analysis
</div>
'''
map3.get_root().html.add_child(folium.Element(title_html))

MousePosition().add_to(map3)

map3.save('folium_map3_proximities.html')
print("\n[DONE] Map 3 saved: folium_map3_proximities.html")
map3

# Cell 8: BONUS MAP - Success Rate Heatmap by Location
print("\n" + "=" * 70)
print("BONUS MAP: Success Rate Analysis")
print("=" * 70)

map_bonus = folium.Map(
    location=[32.0, -100.0],
    zoom_start=4,
    tiles='CartoDB positron'
)

site_stats = df.groupby(['LaunchSite', 'Latitude', 'Longitude']).agg({
    'Class': ['sum', 'count', 'mean']
}).reset_index()

site_stats.columns = ['LaunchSite', 'Latitude', 'Longitude', 'Successes', 'Total', 'Success_Rate']

for idx, row in site_stats.iterrows():
    if row['Success_Rate'] >= 0.8:
        color = 'darkgreen'
    elif row['Success_Rate'] >= 0.6:
        color = 'green'
    elif row['Success_Rate'] >= 0.4:
        color = 'orange'
    else:
        color = 'red'
    
    popup_html = f"""
    <div style="font-family: Arial; font-size: 13px;">
        <b>{row['LaunchSite']}</b><br>
        Total Launches: {row['Total']}<br>
        Successful: {row['Successes']}<br>
        <b style="color: {color};">Success Rate: {row['Success_Rate']*100:.1f}%</b>
    </div>
    """
    
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=row['Total'] * 2,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"{row['LaunchSite']}: {row['Success_Rate']*100:.1f}%",
        color=color,
        fillColor=color,
        fillOpacity=0.6,
        weight=3
    ).add_to(map_bonus)

legend_html = '''
<div style="position: fixed; 
            bottom: 50px; 
            right: 50px; 
            width: 200px; 
            height: 160px; 
            background-color: white; 
            border:2px solid grey; 
            z-index:9999; 
            font-size:13px;
            padding: 10px;
            ">
<b>Success Rate</b><br>
<i class="fa fa-circle" style="color:darkgreen"></i> ≥80%<br>
<i class="fa fa-circle" style="color:green"></i> 60-79%<br>
<i class="fa fa-circle" style="color:orange"></i> 40-59%<br>
<i class="fa fa-circle" style="color:red"></i> <40%<br>
<br>
<small>Circle size = total launches</small>
</div>
'''
map_bonus.get_root().html.add_child(folium.Element(legend_html))

map_bonus.save('folium_map_bonus_success_rate.html')
print("\n[DONE] Bonus map saved: folium_map_bonus_success_rate.html")
map_bonus

# Cell 9: ADDITIONAL MAP - Timeline with Clustering
print("\n" + "=" * 70)
print("ADDITIONAL MAP: Launch Timeline")
print("=" * 70)

map_timeline = folium.Map(
    location=[28.5, -80.6],
    zoom_start=6,
    tiles='OpenStreetMap'
)

marker_cluster = MarkerCluster().add_to(map_timeline)

for idx, row in df.iterrows():
    color = 'green' if row['Class'] == 1 else 'red'
    
    popup_html = f"""
    <div style="font-family: Arial; font-size: 12px; width: 200px;">
        <b>Flight #{row['FlightNumber']}</b><br>
        <hr>
        Date: {row['Date'].strftime('%Y-%m-%d')}<br>
        Site: {row['LaunchSite']}<br>
        Payload: {row['PayloadMass']:.0f} kg<br>
        Orbit: {row['Orbit_clean']}<br>
        Outcome: <span style="color:{color}; font-weight:bold;">
                 {'SUCCESS' if row['Class']==1 else 'FAILURE'}</span>
    </div>
    """
    
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"Flight {row['FlightNumber']} - {row['Date'].strftime('%Y-%m-%d')}",
        icon=folium.Icon(color=color, icon='rocket', prefix='fa')
    ).add_to(marker_cluster)

map_timeline.save('folium_map_timeline_cluster.html')
print("\n[DONE] Timeline map saved: folium_map_timeline_cluster.html")
map_timeline

# Cell 10: Summary
print("\n" + "=" * 70)
print("FOLIUM MAPS SUMMARY")
print("=" * 70)

summary = f"""
INTERACTIVE MAPS CREATED:

1. MAP 1 - Global Launch Sites View (SLIDE 35)
   File: folium_map1_global_sites.html

2. MAP 2 - Launch Outcomes Color-Coded (SLIDE 36)
   File: folium_map2_launch_outcomes.html

3. MAP 3 - Launch Site Proximities (SLIDE 37)
   File: folium_map3_proximities.html

BONUS MAPS:
- Success Rate Heatmap
- Launch Timeline with Clustering

KEY FINDINGS:
- Total unique launch sites: {df['LaunchSite'].nunique()}
- Most used launch site: {df['LaunchSite'].value_counts().index[0]}
- Geographic distribution: East Coast (FL) and West Coast (CA)

DISTANCE ANALYSIS (from KSC LC-39A):
"""

for name, coords in proximities.items():
    distance = geodesic(ksc_location, coords).kilometers
    summary += f"- {name}: {distance:.2f} km\n"

print(summary)

with open('folium_maps_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)

print("\n[DONE] ALL FOLIUM MAPS COMPLETED!")

Libraries imported successfully!
Dataset shape: (179, 14)
Columns: ['FlightNumber', 'Date', 'Year', 'Month', 'Quarter', 'LaunchSite', 'PayloadMass', 'Payload_Category', 'Orbit', 'Orbit_clean', 'MissionOutcome', 'LandingOutcome', 'Class', 'Days_Since_First_Launch']


,FlightNumber,Date,Year,Month,Quarter,LaunchSite,PayloadMass,Payload_Category,Orbit,Orbit_clean,MissionOutcome,LandingOutcome,Class,Days_Since_First_Launch
0,6,2010-06-04 18:45:00+00:00,2010,6,2,CCSFS SLC 40,6630.5,Heavy,LEO,LEO,True,No attempt,0,0
1,7,2010-12-08 15:43:00+00:00,2010,12,4,CCSFS SLC 40,6630.5,Heavy,LEO,LEO,True,No attempt,0,186
2,8,2012-05-22 07:44:00+00:00,2012,5,2,CCSFS SLC 40,525.0,Light,LEO,LEO,True,No attempt,0,717
3,9,2012-10-08 00:35:00+00:00,2012,10,4,CCSFS SLC 40,400.0,Light,ISS,ISS,True,No attempt,0,856
4,10,2013-03-01 19:10:00+00:00,2013,3,1,CCSFS SLC 40,677.0,Light,ISS,ISS,True,No attempt,0,1001



Launch site coordinates:
  CCAFS SLC 40: (28.5621, -80.577)
  CCAFS LC 40: (28.5621, -80.577)
  KSC LC 39A: (28.608, -80.604)
  VAFB SLC 4E: (34.6321, -120.61)
  VAFB SLC 4W: (34.6321, -120.61)

DataFrame with coordinates:


,LaunchSite,Latitude,Longitude
0,CCSFS SLC 40,28.5621,-80.577
1,CCSFS SLC 40,28.5621,-80.577
2,CCSFS SLC 40,28.5621,-80.577
3,CCSFS SLC 40,28.5621,-80.577
4,CCSFS SLC 40,28.5621,-80.577


MAP 1: All Launch Sites - Global View

[DONE] Map 1 saved: folium_map1_global_sites.html
Screenshot this map for SLIDE 35

MAP 2: Launch Outcomes - Color-Coded by Success/Failure

[DONE] Map 2 saved: folium_map2_launch_outcomes.html
Screenshot this map for SLIDE 36

MAP 3: Launch Site Proximities and Distances
Distance to Coastline: 2.35 km (1.46 miles)
Distance to Highway (SR 405): 1.80 km (1.12 miles)
Distance to Railway: 3.23 km (2.01 miles)
Distance to Visitor Complex: 11.98 km (7.44 miles)
Distance to Cape Canaveral SFS: 5.73 km (3.56 miles)

[DONE] Map 3 saved: folium_map3_proximities.html
Screenshot this map for SLIDE 37

BONUS MAP: Success Rate Analysis

[DONE] Bonus map saved: folium_map_bonus_success_rate.html

ADDITIONAL MAP: Launch Timeline

[DONE] Timeline map saved: folium_map_timeline_cluster.html

FOLIUM MAPS SUMMARY

INTERACTIVE MAPS CREATED:

1. MAP 1 - Global Launch Sites View (SLIDE 35)
   File: folium_map1_global_sites.html

2. MAP 2 - Launch Outcomes Color-Coded (